# Objetivo

Revisar o algoritmo de formatação de números

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import re
from pathlib import Path

# 2. Módulos de Terceiros
import numpy as np
import pandas as pd

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
%%html
<style>
table {float:left}
</style>

## Funções auxiliares

In [ ]:
def extract_number(source_number, pattern):

    # Padrões de rejeição
    # mesmo número repetido: 99999999
    # dois números repetidos: 55555999999
    reject_pattern = re.compile(
        r"""
        ^(.)\1+$                    # Ramo 1: Usa Grupo 1 (\1), rejeita mesmo número repetido: 6666666666
        |
        ^(.)\2*(?!\2)(.)\3*$        # Ramo 2: Usa Grupo 2 (\2) e Grupo 3 (\3), rejeita dois números repetidos 5555599999
        """,
        re.VERBOSE
    )
    
    if match := pattern.search(source_number):
        if reject_pattern.match(match.group(match.lastindex)):
            return source_number, False
        return match.group(match.lastindex),True
    
    return source_number, False

## Arquivos de dados

### Dados de origem

In [ ]:
file_numbers = "../assets/number_format_revision.xlsx"
df_numbers = pd.read_excel(file_numbers)
df_numbers = df_numbers.fillna("__NULL__")

df_numbers.head()

# Desenvolvimento

## Estratégias

A partir da revisão dos números anotados, seguir as estratégias:

- Converter números originais para minúsculas
- Remover **apenas** prefixos nacional e internacional (`0` e `00`) do início dos números originais e _filler_ (`f`) do final, não remover caracteres especiais, pois números de serviço da prestadora (`*100`) podem ser confundidos com serviços de utilidade pública (`100`). Não remover o MCC `55` pois pode ser confundido com o código nacional.
- Validar primero números de tamanho igual ao formato esperado, ou seja, números fixos ou móveis **precedidos** do código nacional ou códigos não geográficos (`10` ou `11`), números fixos ou móveis **não precedidos** do código nacional (`8`, ou `9`) e números de serviços de utilidade pública (`3`, `4` ou `5`, `6` ou `7`).
- Regras de validação de números de serviços de utilidade pública devem ser rígidas, ou seja, devem corresponder extamente ao número do serviço e pode ser precedido do código nacional
- Na sequência procurar números que correspondam ao padrão de números fixo, móvel ou CNG no final, início ou em qualquer parte da string, nessa ordem de prioridade. Os números móvel ou fixo podem ser precedidos de indicador de chamada a cobrar (`9090`) ou do código nacional. Avaliar cuidadosamente os resultados.
- Descartar do resultado final números compostos por sequêcia do mesmo caractere

## Padrões

In [ ]:
CLEAN_PATTERN = re.compile(r"^0{1,2}|f$")

In [ ]:
CLEAN_PATTERN.search("00000021")

In [ ]:
#: Resolução 749, art. 15
_CN_PATTERN = r"""
    (?:
        1[1-9]|
        2[12478]|
        3[1-578]|
        4[1-9]|
        5[1345]|
        6[1-9]|
        7[134579]|
        8[1-9]|
        9[1-9]
    )
"""

In [ ]:
# Serviços de 3 dígitos (ex.: 190), sem CN; compõem _SERVICES_PATTERN.
_3_DIGIT_SERVICES_PATTERN = r"""
    (?:
        10[02-6]|
        11[125-8]|
        12[1357-9]|
        13[02-68]|
        14[25-8]|
        15[0-9]|
        16[0-8]|
        18[0158]|
        19[0-9]|
        911
    )
"""

In [ ]:
# Serviços de 4 dígitos (ex.: 1052), sem CN; compõem _SERVICES_PATTERN.
_4_DIGIT_SERVICES_PATTERN = r"""
    (?:
        105[0-35-9]|
        106[013467]|
        133[12]|
        1358|
        1746
    )
"""

In [ ]:
# Extensões do serviço 103 (ex.: 10312), sem CN; compõem _SERVICES_PATTERN.
_103_SERVICES_PATTERN = r"""
    103(?:
        1[2-579]|
        2[13-9]|
        3[124-9]|
        4[1-3578]|
        5[1-468]|
        6[139]|
        8[149]|
        9[168]
    )
"""

In [ ]:
# Extensões do serviço 106 (ex.: 10610), sem CN; compõem _SERVICES_PATTERN.
_106_SERVICES_PATTERN = r"""
    106(?:
        1[0-35-8]|
        2[0145]|
        3[0137]|
        4[37-9]|
        5[0-35]|
        6[016]|
        7[137]|
        8[5-8]|
        9[1359]
    )
"""

In [ ]:
# Composição dos serviços de utilidade pública sem CN
_SERVICES_PATTERN = rf"""
    (?:
        {_3_DIGIT_SERVICES_PATTERN}|
        {_4_DIGIT_SERVICES_PATTERN}|
        {_103_SERVICES_PATTERN}|
        {_106_SERVICES_PATTERN}
    )
"""

In [ ]:
# Resolução 749, art. 11
_STFC_PATTERN = r"[2-6][0-9]{7}"

# Resolução 749 não menciona SME
# Na base da EASI há apenas prefixo com N8=7 designado
_SME_PATTERN = r"7[0-9]{7}"

# Resolução 749, art. 12
# Resolução destina número 7,8 e 9 para N9
# Na base da EASI há apenas N9=9 atribuídos
_SMP_PATTERN = r"9[0-9]{8}"

# Resolução 749, art. 18 (Exceção)
# Ainda há alguns 0800 com 9 dígitos
_SHORT_CNG_PATTERN = r"(?:800[0-9]{6})"

_CNG_PATTERN = r"""
    (?:
        [3589]00[0-9]{7}|
        303[0-9]{7}
    )
"""

In [ ]:
# group(1): apenas o número do serviço, sem o CN
# padrão de números de serviços de utilidade pública deve ser mais
# restritivo que de números fixos ou móveis para minimizar falsos positivos.
# casa somente o número do serviço completo
UTILITY_NUMBERS_PATTERN = re.compile(
    rf"""
    ^
    (?:0?{_CN_PATTERN})?
    ({_SERVICES_PATTERN})
    $
    """,
    re.VERBOSE,
)

In [ ]:
# group(1): fixo ou móvel (8 ou 9 dígitos), sem CN
MID_NUMBERS_PATTERN = re.compile(
    rf"""
    ^(
        {_STFC_PATTERN}|
        {_SME_PATTERN}|
        {_SMP_PATTERN}|
        {_SHORT_CNG_PATTERN}        
    )
    $
    """,
    re.VERBOSE,
)

In [ ]:
# group(1): CN + fixo/móvel, ou CNG
LARGE_NUMBERS_PATTERN = re.compile(
    rf"""
    ^(
        {_CN_PATTERN}
        (?:
            {_STFC_PATTERN}|
            {_SME_PATTERN}|
            {_SMP_PATTERN}
        )|
        {_SHORT_CNG_PATTERN}|
        {_CNG_PATTERN}
    )$
    """,
    re.VERBOSE,
)

In [ ]:
# group(1): CN + fixo/móvel, ou CNG
OTHER_NUMBERS_PATTERN = re.compile(
    rf"""
    (
        {_CN_PATTERN}
        (?:
            {_STFC_PATTERN}|
            {_SME_PATTERN}|
            {_SMP_PATTERN}
        )|
        {_SHORT_CNG_PATTERN}|
        {_CNG_PATTERN}
    )$
    """,
    re.VERBOSE,
)

## Processamento

### Preparação dos dados

In [ ]:
df = df_numbers.copy()
df = df.dropna()
df.head()

### Remoção de prefixos e _fillers_

Remover prefixos e _fillers_ e acrescentar coluna de tamanho após a limpeza para segmentar os próximos passos da análise.

In [ ]:
# Formação de dados para análise da REGEX (exemplo)
numbers = [f"{n}f" for n in df[df["source_number"].str.startswith("0", na=False)]["source_number"]]
numbers = [(n, CLEAN_PATTERN.sub("", n)) for n in numbers]
df_clean = pd.DataFrame(numbers, columns=["source_number", "clean_number"])
df_clean.head()

In [ ]:
# 
df["clean_number"] = df["source_number"].apply(lambda x: CLEAN_PATTERN.sub("",str(x)))
df["number_length"] = df["clean_number"].str.len()
df[df["source_number"].str.startswith("0", na=False)].head()

### Pesquisa segregada por tamanho

#### Segração

In [ ]:
# bins (tamanho)
# 3-7 → SUP
# 8,9 → números fixo/móvel sem CN ou CNG de 9 dígitos
# 10, 11, 12 → números fixo/móvel com CN ou CNG de 9 ou 10 dígitos com ou sem CN
# >= 13 → números sem regra rígida, procurar fixo/móvel/CNG
bins = [0, 3, 8, 10, 12, 50]
df["number_length_bin"] = pd.cut(df["number_length"],bins=bins, right=False)
df.groupby("number_length_bin", observed=True).count()

#### SUP (3, 4, 5, 6, 7)

Números de SUP precedidos ou não pelo `CN`, por exemplo:

| Tamanho    | Exemplo |
|        --: |     --: |
| 3 (sem CN) |     100 |
| 4 (sem CN) |    1052 |
| 5 (sem CN) |   10611 |
| 5 (com CN) |   33190 |
| 6 (com CN) |  271058 |
| 7 (com CN) | 2110621 |

In [ ]:
# 1. Define a máscara de filtragem
mask = df["number_length"].isin([3, 4, 5, 6, 7])

# 2. Aplica a função e desempacota a tupla nas duas novas colunas
df.loc[mask, ["extracted_number", "extracted_number_valid"]] = list(
    df.loc[mask, "clean_number"].apply(
        lambda x: extract_number(x, UTILITY_NUMBERS_PATTERN)
    )
)
df.loc[mask, "number_group"] = "sup"
df[mask].groupby("number_length").sample(5)

#### STFC/SMP/CNG sem CN (8, 9)

Números de telefones fixos, móveis ou de códigos não geográficos **não** precedidos pelo `CN`, por exemplo:

| Tamanho          |   Exemplo |
|              --: |       --: |
| 8 ( STFC sem CN) |  40096700 |
| 9 ( SMP sem CN)  | 992720671 |
| 9 (CNG)          | 800616161 |

In [ ]:
# 1. Define a máscara de filtragem
mask = df["number_length"].isin([8, 9])

# 2. Aplica a função e desempacota a tupla nas duas novas colunas
df.loc[mask, ["extracted_number", "extracted_number_valid"]] = list(
    df.loc[mask, "clean_number"].apply(
        lambda x: extract_number(x, MID_NUMBERS_PATTERN)
    )
)
df.loc[mask, "number_group"] = "stfc_smp_cng_mid"
df[mask].groupby("number_length").sample(5)

#### STFC/SMP com CN e CNG sem CN (10, 11)

Números de telefones fixos e móveis precedidos pelo `CN` e de códigos não geográficos **não** precedidos pelo `CN`, por exemplo:

> _Obsevação:_ Não foram incluídos os números de códigos não geográficos precedidos pelo `CN` para não sobrepor outros números com tamanho `12`.

| Tamanho          |     Exemplo |
|              --: |         --: |
| 8 ( STFC com CN) |  2740096700 |
| 9 ( SMP com CN)  | 31992720671 |
| 9 (CNG)          |  8001575020 |

In [ ]:
# 1. Define a máscara de filtragem
mask = df["number_length"].isin([10, 11])

# 2. Aplica a função e desempacota a tupla nas duas novas colunas
df.loc[mask, ["extracted_number", "extracted_number_valid"]] = list(
    df.loc[mask, "clean_number"].apply(
        lambda x: extract_number(x, LARGE_NUMBERS_PATTERN)
    )
)
df.loc[mask, "number_group"] = "stfc_smp_cng_large"
df[mask].groupby("number_length").sample(5)

In [ ]:
df.loc[df["number_group"].isna(),"number_group"] = "other"
df.pivot_table(index="number_length_bin", columns="number_group", values="source_number", aggfunc="count")

#### Outros tamanhos e situações

##### Padrões

In [ ]:
# group(1): CN + fixo/móvel, ou CNG ao final da string
_TRAILING_NUMBER_PATTERN = re.compile(
    rf"""
    (
        {_CN_PATTERN}
        (?:
            {_STFC_PATTERN}|
            {_SME_PATTERN}|
            {_SMP_PATTERN}
        )|
        {_SHORT_CNG_PATTERN}|
        {_CNG_PATTERN}
    )$
    """,
    re.VERBOSE,
)

In [ ]:
# group(1): CN + fixo/móvel, ou CNG no início da string
# dividr em duas regex: 
_LEADING_NUMBER_PATTERN = re.compile(
    rf"""
    ^(
        {_CN_PATTERN}
        (?:
            {_SMP_PATTERN}|
            {_STFC_PATTERN}|
            {_SME_PATTERN}|
        )|
        {_SHORT_CNG_PATTERN}|
        {_CNG_PATTERN}
    )
    """,
    re.VERBOSE,
)

In [ ]:
_LEADING_NUMBER_PATTERN.search("79933272137")

In [ ]:
# group(1): CN + fixo/móvel, ou CNG em qualquer posição da string
_ANYWHERE_NUMBER_PATTERN = re.compile(
    rf"""
    (
        {_CN_PATTERN}
        (?:
            {_STFC_PATTERN}|
            {_SME_PATTERN}|
            {_SMP_PATTERN}
        )|
        {_SHORT_CNG_PATTERN}|
        {_CNG_PATTERN}
    )
    """,
    re.VERBOSE,
)

In [ ]:
# group(1): retorna [CN + serviço] para 90+CN OU [serviço] para 9090
_NUMBER_PATTERN = rf"(?:{_STFC_PATTERN}|{_SME_PATTERN}|{_SMP_PATTERN})"

# Prefixo 9090 + número (sem CN) -> grupo 1 = número
_PREFIX_9090_PATTERN = re.compile(
    rf"9090({_NUMBER_PATTERN})",
    re.VERBOSE,
)

# Prefixo 90 + CN + número -> grupo 1 = CN + número
_PREFIX_90_PATTERN = re.compile(
    rf"90((?:{_CN_PATTERN}){_NUMBER_PATTERN})",
    re.VERBOSE,
)

_COLLECT_CALL_PATTERN = re.compile(
    rf"90(?:90({_NUMBER_PATTERN})|((?:{_CN_PATTERN}){_NUMBER_PATTERN}))",
    re.VERBOSE,
)
# grupo 1 = caso 9090, grupo 2 = caso 90

In [ ]:
def re_match(pattern, x):
    if m:= pattern.match(x):
        return m.group(m.lastindex)
    return np.nan

def re_search(pattern, x):
    if m:= pattern.search(x):
        return m.group(m.lastindex)
    return np.nan

In [ ]:
_COLLECT_CALL_PATTERN = rf"90(?:90({_NUMBER_PATTERN})|((?:{_CN_PATTERN}){_NUMBER_PATTERN}))"

In [ ]:
# 1. Padrão de chamada a cobrar: consome 9090 ou 90+CN sem capturar o prefixo a cobrar
_COLLECT_CALL_PATTERN = re.compile(
    rf"""
        (?:
            9090(?:{_NUMBER_PATTERN})
            90(?:{_CN_PATTERN})(?:{_NUMBER_PATTERN})
        )
    """, 
        re.VERBOSE
)

# 2. Trailing pattern: abre o group(1) exatamente após consumir o 90 ou 9090
_TRAILING_NUMBER_PATTERN = re.compile(
    rf"""
    (?:
        # Caso 1: Chamada a cobrar (consome 9090 ou 90+CN e captura só o destino no group 1)
        9090((?:{_NUMBER_PATTERN})) |
        90((?:{_CN_PATTERN})(?:{_NUMBER_PATTERN}))|
        
        # Caso 2: Demais padrões normais no final da string
        (
            (?:{_CN_PATTERN})(?:{_STFC_PATTERN}|{_SME_PATTERN}|{_SMP_PATTERN})|
            (?:{_SHORT_CNG_PATTERN})|
            (?:{_CNG_PATTERN})
        )
    ) $
    """,
    re.VERBOSE,
)

_TRAILING_NUMBER_PATTERN = re.compile(
    rf"""
    (
        {_CN_PATTERN} {_NUMBER_PATTERN} |
        (?<=9090) {_NUMBER_PATTERN} |
        {_SHORT_CNG_PATTERN} |
        {_CNG_PATTERN}
    ) $
    """,
    re.VERBOSE,
)

In [ ]:
_TRAILING_NUMBER_PATTERN.search("909099115446729090991544672").group(1)

In [ ]:
m = _TRAILING_NUMBER_PATTERN.search("1232732544672")
m.group(m.lastindex)

##### Extração

In [ ]:
df.loc[:,"collect_call_match"] = df["clean_number"].apply(lambda x: re_search(_COLLECT_CALL_PATTERN,x))
df.loc[:,"trailing_match"] = df["clean_number"].apply(lambda x: re_search(_TRAILING_NUMBER_PATTERN,x))
df.loc[:,"leading_match"] = df["clean_number"].apply(lambda x: re_match(_LEADING_NUMBER_PATTERN,x))
df.loc[:,"anywhere_match"] = df["clean_number"].apply(lambda x: re_search(_ANYWHERE_NUMBER_PATTERN,x))
df.head()

In [ ]:
df[df["clean_number"].str.contains("9090")]["clean_number"].head()

**Atenção ao exemplo**. O _match_ correto de `55034199909051982169391` parece ser o _trailing_match_ pois, aparentemente se trata de uma chamada a cobrar e há, aparentemente, um CN no meio do `9090` e do número de telefone. Nesse cenário, _trailing_match_ deve ter precedência sobre _collect_call_match_.

### Resultado

In [ ]:
df.head()

In [ ]:
df.to_excel("output_numbers.xlsx")

Sequência a aplicar e testar:

- por tamanho
- trailing (inclusive collect call)
- leading
- anywhere 
- 

Até onde vi não faz sentido pegar chamadas a cobrar em qualquer ponto, o melhor resultado foi no final da string.

Exemplos: 

| nu_destino |
| --- |
| `5527#9090999788442` |
| `5586#1#9090981785314` |
| `5592#9090991655077` |
| `5592#9090992132032` |
| `5562#9090991814003` |
| `5511#9090997901481` |
| `5522#9090999589018` |
| `5579#9090999106439` |
| `5565#9090996920435` |
| `5579#9090998615222` |

In [ ]:
# df[df["clean_number"].str.contains("9090")]

## Análises EASI

In [ ]:
read_easi_file = lambda file: pd.read_csv(file, sep=";", encoding="latin1", dtype="str")

#### Análise CNG

In [ ]:
cng_folder = Path("../assets/easi/cng")
cng_files = [file for file in cng_folder.glob("*.zip")]

df_cng = pd.concat([read_easi_file(file) for file in cng_files])

df_cng["Tamanho"] = df_cng["Código Não Geográfico"].str.len()
df_cng["Prefixo"] = df_cng["Código Não Geográfico"].str.slice(stop=3)
df_cng.head()

In [ ]:
df_cng.pivot_table(index="Prefixo",columns="Tamanho",values="Status",aggfunc="count").fillna(0).astype(int)

#### Análise SMP

In [ ]:
smp_folder = Path("../assets/easi/smp")
smp_files = [file for file in smp_folder.glob("*.zip")]

df_smp = pd.concat([read_easi_file(file) for file in smp_files])
df_smp["N9"] = df_smp["Prefixo"].str.slice(stop=1)

df_smp.head()

In [ ]:
df_smp["N9"].value_counts()

In [ ]:
df_smp[df_smp["N9"]=="7"].head()

# Testes

In [ ]:
from teleutils.preprocessing.number_format import extract_number

In [ ]:
file_numbers = "/workspaces/teleutils/tests/assets/number_format_revision.xlsx"
df_tests = pd.read_excel(file_numbers,sheet_name="tests")
df_tests = df_tests.fillna("__NULL__")
df_tests["expected_formated_number"] = df_tests["expected_formated_number"].astype(str).str.strip()

df_tests.head() 

In [ ]:
df_tests.loc[:, ["extracted_number","extracted_method", "extracted_number_valid"]] = list(df_tests.source_number.apply(lambda x: extract_number(x)))
df_tests.head()

In [ ]:
df_tests["extraction_test_passed"] = df_tests["extracted_number"]==df_tests["expected_formated_number"]
df_tests.groupby("extraction_test_passed").count()

In [ ]:
df_tests[df_tests["extraction_test_passed"]==False].shape

In [ ]:
df_tests[df_tests["extraction_test_passed"]==False]

In [ ]:
# print(df_tests[df_tests["extraction_test_passed"]==False].to_string())